In [6]:
# NB01_data_preparation.ipynb
# Study 2: Uncovering Hidden Evaluator Judgment Patterns in Credit Rating Overrides
#
# Differences from Study 1
# ┌──────────────────┬──────────────────────────────┬─────────────────────────────────────────┐
# │ Item             │ Study 1                      │ Study 2 (this notebook)                 │
# ├──────────────────┼──────────────────────────────┼─────────────────────────────────────────┤
# │ Unit of analysis │ All firms (override vs. not) │ Upgrade-overridden firms only           │
# │ Added label      │ override_flag, grade_diff    │ group (risky_upgrade / safe_upgrade)    │
# └──────────────────┴──────────────────────────────┴─────────────────────────────────────────┘
#
# Outputs
#   data/processed/polish_master.parquet   — compatible with Study 1 (N = 43,405)
#   data/processed/upgrade_cohort.parquet  — Study 2 cohort with group labels
#
# Download data: https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data
# Place 1year.arff ~ 5year.arff in data/raw/ before running.

import os
import warnings
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

RAW_DIR     = "../data/raw/"
PROC_DIR    = "../data/processed/"
RANDOM_SEED = 42

os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)
np.random.seed(RANDOM_SEED)

print("Directories ready.")


# ── 1. Load .arff files ───────────────────────────────────────────────────────

ARFF_FILES = {
    f"{i}year": os.path.join(RAW_DIR, f"{i}year.arff")
    for i in range(1, 6)
}

frames = []
for year_key, fpath in ARFF_FILES.items():
    if not os.path.exists(fpath):
        print(f"[SKIP] {fpath} not found.")
        continue
    data, meta = arff.loadarff(fpath)
    df = pd.DataFrame(data)
    df = df.applymap(lambda x: x.decode() if isinstance(x, bytes) else x)
    df["year_horizon"] = int(year_key[0])
    frames.append(df)
    print(f"Loaded {year_key}: {len(df):,} rows, {df.shape[1]} cols")

if not frames:
    raise FileNotFoundError(
        "No .arff files found. Download from UCI and place in data/raw/"
    )

master = pd.concat(frames, ignore_index=True)
print(f"\nMaster shape: {master.shape}")


# ── 2. Target variable & column renaming ─────────────────────────────────────

master["default"] = (
    master["class"].astype(str).str.strip().isin(["1", "1.0", "b'1'"])
).astype(int)

rename_map = {
    "Attr1" : "net_profit_to_assets",
    "Attr2" : "total_liabilities_to_assets",
    "Attr3" : "working_capital_to_assets",
    "Attr4" : "current_assets_to_short_liabilities",
    "Attr5" : "cash_to_current_liabilities",
    "Attr6" : "retained_earnings_to_assets",
    "Attr7" : "ebit_to_assets",
    "Attr8" : "book_value_equity_to_liabilities",
    "Attr9" : "sales_to_assets",
    "Attr10": "equity_to_assets",
}
master.rename(columns=rename_map, inplace=True)

print(f"Overall default rate: {master['default'].mean():.4f}  ({master['default'].mean()*100:.2f}%)")


# ── 3. Outlier clipping (1st / 99th percentile) ───────────────────────────────

numeric_cols = master.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ("default", "year_horizon")]

for col in numeric_cols:
    lo, hi = master[col].quantile([0.01, 0.99])
    master[col] = master[col].clip(lo, hi)

print(f"Clipped {len(numeric_cols)} numeric columns.")


# ── 4. Missing value imputation ───────────────────────────────────────────────

missing     = master.isnull().sum()
missing_pct = (missing / len(master) * 100).round(2)
missing_report = pd.DataFrame({"missing_n": missing, "missing_pct": missing_pct})
missing_report = (
    missing_report[missing_report["missing_n"] > 0]
    .sort_values("missing_pct", ascending=False)
)
print(f"Columns with missing values: {len(missing_report)}")

for col in numeric_cols:
    if master[col].isnull().any():
        master[col].fillna(master[col].median(), inplace=True)

print(f"Remaining missing values: {master.isnull().sum().sum()}")


# ── 5. System credit grading model (identical parameters to Study 1) ─────────
#
# Using the exact same configuration as Study 1 ensures that pd_system and
# grade_ordinal values are consistent, so Study 2 results are directly
# comparable to and continuous with Study 1.

FEATURE_COLS = [c for c in numeric_cols if c != "default"]
X            = master[FEATURE_COLS].values
y            = master["default"].values
X_scaled     = StandardScaler().fit_transform(X)

lr = LogisticRegression(C=0.1, class_weight="balanced",
                        max_iter=1000, random_state=RANDOM_SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

pd_system           = cross_val_predict(lr, X_scaled, y, cv=cv, method="predict_proba")[:, 1]
master["pd_system"] = pd_system

print(f"\nSystem model ROC-AUC (5-fold OOF): {roc_auc_score(y, pd_system):.4f}")


# ── 6. Grade assignment — quantile-based 7-grade scale (identical to Study 1) ─
#
# Grades are assigned by quantile so each bucket contains roughly equal numbers
# of observations (~6,200 each).
#
# pd_system is a probability of default: higher value = riskier firm.
# Grade ordering: AAA (safest, lowest pd) → CCC (riskiest, highest pd).
# Therefore the lowest pd_system quantile bin maps to AAA and the highest to CCC.
#
# pd.cut bins are defined in ascending order of pd_system.
# We assign labels in ascending risk order: AAA, AA, A, BBB, BB, B, CCC
# so that the first (lowest pd) bin receives AAA and the last (highest pd) bin
# receives CCC — which is the correct credit-rating convention.

GRADE_LABELS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]
GRADE_NUM    = {"AAA": 7, "AA": 6, "A": 5, "BBB": 4, "BB": 3, "B": 2, "CCC": 1}

quantile_cuts     = np.quantile(master["pd_system"], np.linspace(0, 1, 8))
quantile_cuts[0]  = -np.inf
quantile_cuts[-1] =  np.inf

# Labels are supplied in low-pd → high-pd order (AAA first, CCC last),
# matching the ascending bin boundaries produced by np.quantile.
master["system_grade"] = pd.cut(
    master["pd_system"],
    bins=quantile_cuts,
    labels=GRADE_LABELS,        # AAA = lowest pd bin, CCC = highest pd bin
    include_lowest=True
)
master["grade_ordinal"] = master["system_grade"].map(GRADE_NUM).astype(int)

print("\nGrade distribution (target: ~6,200 per grade):")
print(master["system_grade"].value_counts().sort_index())
print("\nRealised default rate by system grade (must increase AAA → CCC):")
print(master.groupby("system_grade", observed=True)["default"].mean().round(4))

# Sanity check: verify monotonic ordering of default rates
dr_by_grade = (
    master.groupby("system_grade", observed=True)["default"]
    .mean()
    .reindex(GRADE_LABELS)      # AAA → CCC order
)
assert list(dr_by_grade) == sorted(dr_by_grade), (
    "Default rates are not monotonically increasing from AAA to CCC. "
    "Check grade label assignment."
)
print("  [OK] Default rates increase monotonically from AAA to CCC.")


# ── 7. Override simulation — Base scenario (identical to Study 1, Table 4) ────
#
# Upgrade probabilities are concentrated in lower (riskier) grades to reflect
# the optimistic evaluator bias documented in Korean credit rating practice.
# Downgrade probabilities reproduce the Study 1 Base scenario aggregate split:
# approximately 14% upgrade, 21% downgrade, 65% maintain.
#
# grade_diff convention (consistent with Study 1):
#   -1 = upgrade   (evaluator assigns a better grade than the system)
#    0 = maintain
#   +1 = downgrade (evaluator assigns a worse grade than the system)

UPGRADE_PROB = {
    "CCC": 0.80, "B": 0.70, "BB": 0.50,
    "BBB": 0.35, "A": 0.20, "AA": 0.00, "AAA": 0.00,
}
DOWNGRADE_PROB = {
    "CCC": 0.00, "B": 0.10, "BB": 0.15,
    "BBB": 0.20, "A": 0.25, "AA": 0.30, "AAA": 0.35,
}

rng         = np.random.default_rng(RANDOM_SEED)
grade_diffs = []

for grade in master["system_grade"].astype(str):
    g_num  = GRADE_NUM.get(grade, 4)
    up_p   = UPGRADE_PROB.get(grade, 0.0)
    down_p = DOWNGRADE_PROB.get(grade, 0.0)
    r      = rng.random()
    if r < up_p and g_num < 7:
        grade_diffs.append(-1)          # upgrade
    elif r < up_p + down_p and g_num > 1:
        grade_diffs.append(1)           # downgrade
    else:
        grade_diffs.append(0)           # maintain

master["grade_diff"]          = np.array(grade_diffs, dtype=int)
master["override_flag"]       = (master["grade_diff"] != 0).astype(int)
master["final_grade_ordinal"] = master["grade_ordinal"].astype(int) - master["grade_diff"]

direction_map = {-1: "Upgrade", 0: "Maintain", 1: "Downgrade"}
print("\nDefault rate by override direction (replicating Study 1, Table 7):")
print(
    master.groupby(master["grade_diff"].map(direction_map))["default"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "default_rate", "count": "n"})
    .round(4)
)
print("  Study 1 reference: Upgrade 7.69%, Maintain 4.86%, Downgrade 2.70%")


# ── 8. [Study 2 — NEW] Cohort extraction and group labelling ─────────────────
#
# Study 1 analysed all 43,405 observations with override direction as the
# independent variable.
# Study 2 restricts the sample to upgrade-overridden firms only and asks:
# within that group, what financial characteristics distinguish firms that
# subsequently defaulted from those that survived?
#
# group column:
#   'risky_upgrade' — upgraded then defaulted  → comparison group (evaluator wrong)
#   'safe_upgrade'  — upgraded then survived   → control group   (evaluator correct)

INV_GRADE_NUM  = {v: k for k, v in GRADE_NUM.items()}
upgrade_cohort = master[master["grade_diff"] == -1].copy()

upgrade_cohort["group"]    = np.where(upgrade_cohort["default"] == 1,
                                       "risky_upgrade",
                                       "safe_upgrade")
upgrade_cohort["is_risky"] = (upgrade_cohort["group"] == "risky_upgrade").astype(int)

# Grade transition features
upgrade_cohort["from_grade"] = upgrade_cohort["grade_ordinal"].map(INV_GRADE_NUM)
upgrade_cohort["to_grade"]   = upgrade_cohort["final_grade_ordinal"].map(INV_GRADE_NUM)
upgrade_cohort["transition"] = upgrade_cohort["from_grade"] + "→" + upgrade_cohort["to_grade"]

# High-risk transition flag: Study 1 (Table 12) showed CCC- and B-origin
# transitions carry substantially elevated default rates
upgrade_cohort["high_risk_transition"] = (
    upgrade_cohort["from_grade"].isin(["CCC", "B"])
).astype(int)

risky_n = (upgrade_cohort["group"] == "risky_upgrade").sum()
safe_n  = (upgrade_cohort["group"] == "safe_upgrade").sum()

print("\n=== Study 2 Analysis Cohort ===")
print(f"Total upgrade-overridden observations : {len(upgrade_cohort):,}")
print(f"  risky_upgrade (comparison group)   : {risky_n:,}  ({risky_n/len(upgrade_cohort)*100:.1f}%)")
print(f"  safe_upgrade  (control group)      : {safe_n:,}  ({safe_n/len(upgrade_cohort)*100:.1f}%)")
print(f"  Post-upgrade default rate          : {upgrade_cohort['default'].mean()*100:.2f}%")

print("\nDefault rate by grade transition (n >= 20):")
print(
    upgrade_cohort.groupby("transition")["default"]
    .agg(["mean", "count"])
    .query("count >= 20")
    .sort_values("mean", ascending=False)
    .rename(columns={"mean": "default_rate", "count": "n"})
    .assign(default_rate=lambda x: (x["default_rate"] * 100).round(2))
)

hr_dr  = upgrade_cohort[upgrade_cohort["high_risk_transition"] == 1]["default"].mean() * 100
low_dr = upgrade_cohort[upgrade_cohort["high_risk_transition"] == 0]["default"].mean() * 100
print(f"\nDefault rate — high-risk transitions (CCC/B origin) : {hr_dr:.2f}%")
print(f"Default rate — other transitions                     : {low_dr:.2f}%")


# ── 9. Save ───────────────────────────────────────────────────────────────────

# Study 1-compatible master file
master_path = os.path.join(PROC_DIR, "polish_master.parquet")
master.to_parquet(master_path, index=False)
print(f"\nSaved: {master_path}  |  Shape: {master.shape}")

# Study 2 upgrade cohort
cohort_path = os.path.join(PROC_DIR, "upgrade_cohort.parquet")
upgrade_cohort.to_parquet(cohort_path, index=False)
print(f"Saved: {cohort_path}  |  Shape: {upgrade_cohort.shape}")

print("\n[New columns added for Study 2]")
for c in ["group", "is_risky", "from_grade", "to_grade",
          "transition", "high_risk_transition"]:
    print(f"  + {c}")

print("\nNext step → NB02_eda_descriptive.ipynb")

Directories ready.
Loaded 1year: 7,027 rows, 66 cols
Loaded 2year: 10,173 rows, 66 cols
Loaded 3year: 10,503 rows, 66 cols
Loaded 4year: 9,792 rows, 66 cols
Loaded 5year: 5,910 rows, 66 cols

Master shape: (43405, 66)
Overall default rate: 0.0482  (4.82%)
Clipped 64 numeric columns.
Columns with missing values: 64
Remaining missing values: 0

System model ROC-AUC (5-fold OOF): 0.7782

Grade distribution (target: ~6,200 per grade):
system_grade
AAA    6201
AA     6201
A      6200
BBB    6201
BB     6200
B      6201
CCC    6201
Name: count, dtype: int64

Realised default rate by system grade (must increase AAA → CCC):
system_grade
AAA    0.0081
AA     0.0113
A      0.0179
BBB    0.0250
BB     0.0413
B      0.0745
CCC    0.1592
Name: default, dtype: float64
  [OK] Default rates increase monotonically from AAA to CCC.

Default rate by override direction (replicating Study 1, Table 7):
            default_rate      n
grade_diff                     
Downgrade         0.0228   8411
Maintain  